In [20]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from imblearn.over_sampling import SMOTE

In [2]:
fp = r"C:\Users\balaj\OneDrive\Desktop\disease prediction models\good dataset\chronic_kidney.csv".replace("\\","/")
df = pd.read_csv(fp)

In [3]:
df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,1,1,1,1,121.0,...,44.0,7800.0,5.2,1,1,1,1,1,1,1
1,7.0,50.0,1.020,4.0,0.0,1,1,1,1,99.0,...,38.0,6000.0,5.2,2,2,1,1,1,1,1
2,62.0,80.0,1.010,2.0,3.0,1,1,1,1,423.0,...,31.0,7500.0,5.2,2,1,1,2,1,2,1
3,48.0,70.0,1.005,4.0,0.0,1,2,2,1,117.0,...,32.0,6700.0,3.9,1,2,1,2,2,2,1
4,51.0,80.0,1.010,2.0,0.0,1,1,1,1,106.0,...,35.0,7300.0,4.6,2,2,1,1,1,1,1


In [4]:
df['class'].value_counts()

class
1    250
2    149
Name: count, dtype: int64

In [5]:
df.shape

(399, 25)

In [6]:
x = df.drop('class', axis=1)
y = df['class']

In [8]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
smote = SMOTE(random_state=42)

x_train_smote, y_train_smote = smote.fit_resample(
    x_train,
    y_train
)

In [12]:
print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
class
1    200
2    119
Name: count, dtype: int64

After SMOTE:
class
2    200
1    200
Name: count, dtype: int64


In [13]:
def run_models_with_cv(X, y, folds=5):
    
    # Keeps target-class ratio similar in each fold
    cv = StratifiedKFold(
        n_splits=folds,
        shuffle=True,
        random_state=42
    )
    
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000))
        ]),
        
        "Decision Tree": DecisionTreeClassifier(
            random_state=42
        ),
        
        "Random Forest": RandomForestClassifier(
            random_state=42
        ),
        
        "Gradient Boosting": GradientBoostingClassifier(
            random_state=42
        ),
        
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC())
        ]),
        
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        
        "Naive Bayes": Pipeline([
            ("scaler", StandardScaler()),
            ("model", GaussianNB())
        ])
    }
    
    results = []
    
    for name, model in models.items():
        
        scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="accuracy"
        )
        
        results.append({
            "Model": name,
            "Mean Accuracy (%)": scores.mean() * 100,
            "Std (%)": scores.std() * 100,
            "Fold Scores (%)": np.round(scores * 100, 2)
        })
        
        print("=" * 55)
        print(f"Model: {name}")
        print(f"Fold Accuracies: {np.round(scores * 100, 2)}")
        print(f"Mean Accuracy: {scores.mean() * 100:.2f}%")
        print(f"Standard Deviation: {scores.std() * 100:.2f}%")
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(
        by="Mean Accuracy (%)",
        ascending=False
    ).reset_index(drop=True)
    
    return results_df

In [14]:
run_models_with_cv(x_train_smote, y_train_smote)

Model: Logistic Regression
Fold Accuracies: [ 98.75 100.   100.    97.5  100.  ]
Mean Accuracy: 99.25%
Standard Deviation: 1.00%
Model: Decision Tree
Fold Accuracies: [97.5  95.   98.75 91.25 95.  ]
Mean Accuracy: 95.50%
Standard Deviation: 2.57%
Model: Random Forest
Fold Accuracies: [100.    98.75 100.   100.   100.  ]
Mean Accuracy: 99.75%
Standard Deviation: 0.50%
Model: Gradient Boosting
Fold Accuracies: [100.   97.5 100.  100.  100. ]
Mean Accuracy: 99.50%
Standard Deviation: 1.00%
Model: SVM
Fold Accuracies: [ 98.75  97.5  100.    97.5  100.  ]
Mean Accuracy: 98.75%
Standard Deviation: 1.12%
Model: KNN
Fold Accuracies: [97.5  96.25 95.   93.75 97.5 ]
Mean Accuracy: 96.00%
Standard Deviation: 1.46%
Model: Naive Bayes
Fold Accuracies: [98.75 98.75 97.5  96.25 93.75]
Mean Accuracy: 97.00%
Standard Deviation: 1.87%


,Model,Mean Accuracy (%),Std (%),Fold Scores (%)
0,Random Forest,99.75,0.500000,"[100.0, 98.75, 100.0, 100.0, 100.0]"
1,Gradient Boosting,99.50,1.000000,"[100.0, 97.5, 100.0, 100.0, 100.0]"
2,Logistic Regression,99.25,1.000000,"[98.75, 100.0, 100.0, 97.5, 100.0]"
3,SVM,98.75,1.118034,"[98.75, 97.5, 100.0, 97.5, 100.0]"
4,Naive Bayes,97.00,1.870829,"[98.75, 98.75, 97.5, 96.25, 93.75]"
5,KNN,96.00,1.457738,"[97.5, 96.25, 95.0, 93.75, 97.5]"
6,Decision Tree,95.50,2.573908,"[97.5, 95.0, 98.75, 91.25, 95.0]"


In [24]:
def run_models_train_test_table(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    x_train_smote, y_train_smote = smote.fit_resample(
        x_train,
        y_train
    )
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000))
        ]),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC())
        ]),
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        "Naive Bayes": Pipeline([
            ("scaler", StandardScaler()),
            ("model", GaussianNB())
        ])
    }

    results = []

    for name, model in models.items():
        model.fit( x_train_smote, y_train_smote)

        train_pred = model.predict(x_train_smote)
        test_pred = model.predict(X_test)

        train_acc = accuracy_score(y_train_smote, train_pred) * 100
        test_acc = accuracy_score(y_test, test_pred) * 100

        results.append({
            "Model": name,
            "Train Accuracy (%)": train_acc,
            "Test Accuracy (%)": test_acc,
            "Gap (%)": train_acc - test_acc
        })

    results_df = pd.DataFrame(results).sort_values(
        by="Test Accuracy (%)",
        ascending=False
    ).reset_index(drop=True)

    return results_df

In [25]:
run_models_train_test_table(x, y)

,Model,Train Accuracy (%),Test Accuracy (%),Gap (%)
0,Random Forest,100.00,100.00,0.00
1,SVM,99.75,100.00,-0.25
2,Gradient Boosting,100.00,100.00,0.00
3,Logistic Regression,99.75,98.75,1.00
4,Naive Bayes,97.00,97.50,-0.50
5,KNN,97.75,96.25,1.50
6,Decision Tree,100.00,93.75,6.25


In [18]:
def run_all_models_metrics(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000))
        ]),
        "Decision Tree": DecisionTreeClassifier(random_state=random_state),
        "Random Forest": RandomForestClassifier(random_state=random_state),
        "Gradient Boosting": GradientBoostingClassifier(random_state=random_state),
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(probability=True))
        ]),
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        "Naive Bayes": Pipeline([
            ("scaler", StandardScaler()),
            ("model", GaussianNB())
        ])
    }

    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        train_acc = accuracy_score(y_train, train_pred) * 100
        test_acc = accuracy_score(y_test, test_pred) * 100
        gap = train_acc - test_acc

        rec = recall_score(y_test, test_pred, zero_division=0)
        f1 = f1_score(y_test, test_pred, zero_division=0)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:, 1]
        else:
            y_prob = model.decision_function(X_test)

        roc_auc = roc_auc_score(y_test, y_prob) * 100

        results.append({
            "Model": name,
            "Accuracy (%)": test_acc,
            "Recall": rec,
            "F1": f1,
            "ROC-AUC (%)": roc_auc,
            "Gap (%)": gap
        })

    results_df = pd.DataFrame(results).sort_values(
        by="Accuracy (%)",
        ascending=False
    ).reset_index(drop=True)

    return results_df

In [21]:
run_all_models_metrics(x,y)

C:\Users\balaj\AppData\Roaming\Python\Python311\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Accuracy (%),Recall,F1,ROC-AUC (%),Gap (%)
0,Random Forest,100.00,1.00,1.000000,100.000000,0.000000
1,SVM,100.00,1.00,1.000000,100.000000,-0.626959
2,Gradient Boosting,100.00,1.00,1.000000,100.000000,0.000000
3,Logistic Regression,98.75,0.98,0.989899,100.000000,0.936520
4,KNN,97.50,0.96,0.979592,98.933333,0.305643
5,Naive Bayes,97.50,0.96,0.979592,98.000000,-1.261755
6,Decision Tree,93.75,1.00,0.952381,91.666667,6.250000


In [23]:
df.columns

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'class'],
      dtype='object')